In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from mpl_toolkits.mplot3d import Axes3D


def rossler4_rhs(t, state, a=0.25, b=3.0, c=0.5, d=0.05):
    x, y, z, w = state
    dx = -(y + z)
    dy = x + a * y + w
    dz = b + x * z
    dw = -c * z + d * w
    return [dx, dy, dz, dw]


# Integreren (sneller: minder punten, soepelere toleranties)
a, b, c, d = 0.25, 3.0, 0.5, 0.05
s0 = [1.0, 0.0, 0.0, 0.0]
t_span = (0.0, 400.0)
t_eval = np.linspace(*t_span, 50_000)

sol = solve_ivp(
    rossler4_rhs,
    t_span,
    s0,
    t_eval=t_eval,
    args=(a, b, c, d),
    rtol=1e-6,
    atol=1e-9,
    max_step=0.05,
)
x, y, z, w = sol.y

# Transient weggooien
skip = int(0.2 * x.size)
x, y, z, w = x[skip:], y[skip:], z[skip:], w[skip:]

# 3D + kleur = w
fig = plt.figure(figsize=(8, 6))
ax: Axes3D = fig.add_subplot(111, projection="3d")
p = ax.scatter(x[::10], y[::10], z[::10], c=w[::10], s=0.2)  # downsample voor snelheid
fig.colorbar(p, ax=ax, shrink=0.6, label="w")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")
ax.set_title("4D Rössler: (x,y,z) met w als kleur")
plt.show()


In [ ]:
pairs = [("x","y", x,y), ("x","z", x,z), ("x","w", x,w),
         ("y","z", y,z), ("y","w", y,w), ("z","w", z,w)]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, (n1, n2, u, v) in zip(axes.ravel(), pairs):
    ax.plot(u[::10], v[::10], lw=0.2)
    ax.set_xlabel(n1); ax.set_ylabel(n2)
    ax.grid(True, alpha=0.3)
fig.suptitle("4D Rössler: 2D projecties (downsampled)")
plt.tight_layout()
plt.show()


In [ ]:
# Poincaré: w = 0 crossing met dw/dt > 0
w_series = w
dw_series = np.gradient(w_series, sol.t[skip:])  # ruwe dw/dt

cross = np.where((w_series[:-1] < 0) & (w_series[1:] >= 0) & (dw_series[1:] > 0))[0]
# lineaire interpolatie naar w=0
alpha = -w_series[cross] / (w_series[cross+1] - w_series[cross])
xp = x[cross] + alpha*(x[cross+1]-x[cross])
yp = y[cross] + alpha*(y[cross+1]-y[cross])
zp = z[cross] + alpha*(z[cross+1]-z[cross])

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(xp, yp, zp, s=2, alpha=0.6)
ax.set_xlabel("x"); ax.set_ylabel("y"); ax.set_zlabel("z")
ax.set_title("Poincaré sectie: w=0, dw/dt>0")
plt.show()


In [ ]:
X = np.vstack([x, y, z, w]).T
X = X - X.mean(axis=0, keepdims=True)

# PCA via SVD
U, S, Vt = np.linalg.svd(X, full_matrices=False)
PC = X @ Vt.T  # kolommen = principal components

plt.figure(figsize=(6,6))
plt.plot(PC[::10,0], PC[::10,1], lw=0.2)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.grid(True, alpha=0.3)
plt.title("PCA projectie van 4D attractor naar 2D")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from scipy.integrate import solve_ivp


# -------------------------
# 1) 4D Rössler hyperchaos RHS
# -------------------------
def rossler4_rhs(t, state, a=0.25, b=3.0, c=0.5, d=0.05):
    x, y, z, w = state
    dx = -(y + z)
    dy = x + a * y + w
    dz = b + x * z
    dw = -c * z + d * w
    return [dx, dy, dz, dw]


# -------------------------
# 2) Integratie helper
# -------------------------
def integrate_rossler4(
    t_span=(0.0, 400.0),
    n=80_000,
    s0=(1.0, 0.0, 0.0, 0.0),
    a=0.25, b=3.0, c=0.5, d=0.05,
    rtol=1e-9, atol=1e-12,
):
    t_eval = np.linspace(t_span[0], t_span[1], n)
    sol = solve_ivp(
        rossler4_rhs, t_span, s0,
        t_eval=t_eval, args=(a, b, c, d),
        rtol=rtol, atol=atol
    )
    x, y, z, w = sol.y
    return sol.t, x, y, z, w


# -------------------------
# 3) Animatie: 3D window, kleur = w
# -------------------------
def animate_rossler4(
    t, x, y, z, w,
    skip_first_frac=0.2,
    window=4000,        # aantal punten in het "lopende venster"
    frame_step=50,      # hoeveel punten je per frame opschuift
    interval_ms=30,     # tijd tussen frames (ms)
    stride=2,           # downsample binnen het venster (snelheid)
    fixed_limits=True,  # vaste assen (rustiger) vs meeschuivend
    elev=25, azim=-60,  # camera
    save_path=None,     # bv "rossler4.mp4" of "rossler4.gif"
    fps=30,
):
    # ---- transient skip ----
    n0 = int(skip_first_frac * len(t))
    t, x, y, z, w = t[n0:], x[n0:], y[n0:], z[n0:], w[n0:]

    # sanity
    if window >= len(t) - 2:
        raise ValueError("window is te groot tov het aantal punten (na transient-skip).")

    # vaste limieten (optioneel)
    if fixed_limits:
        xmin, xmax = np.min(x), np.max(x)
        ymin, ymax = np.min(y), np.max(y)
        zmin, zmax = np.min(z), np.max(z)
        # kleine padding
        pad = 0.03
        xr = xmax - xmin if xmax > xmin else 1.0
        yr = ymax - ymin if ymax > ymin else 1.0
        zr = zmax - zmin if zmax > zmin else 1.0
        xmin -= pad * xr; xmax += pad * xr
        ymin -= pad * yr; ymax += pad * yr
        zmin -= pad * zr; zmax += pad * zr

    # kleur-normalisatie over alles (na transient)
    norm = Normalize(vmin=np.min(w), vmax=np.max(w))

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection="3d")
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    title = ax.set_title("4D Rössler: (x,y,z) met w als kleur — moving window")

    # placeholder: Line3DCollection (gekleurd per segment)
    lc = Line3DCollection([], cmap="viridis", norm=norm, linewidth=1.0)
    ax.add_collection3d(lc)

    # colorbar
    mappable = plt.cm.ScalarMappable(norm=norm, cmap="viridis")
    mappable.set_array(w)
    cbar = fig.colorbar(mappable, ax=ax, shrink=0.6, pad=0.1)
    cbar.set_label("w")

    if fixed_limits:
        ax.set_xlim(xmin, xmax)
        ax.set_ylim(ymin, ymax)
        ax.set_zlim(zmin, zmax)

    # frames: start-index van elk window
    starts = np.arange(0, len(t) - window - 1, frame_step)

    def make_segments(xw, yw, zw):
        pts = np.column_stack([xw, yw, zw])
        # segmenten: (N-1, 2, 3)
        return np.stack([pts[:-1], pts[1:]], axis=1)

    def update(i):
        start_idx = starts[i]
        end_idx = start_idx + window

        xw = x[start_idx:end_idx:stride]
        yw = y[start_idx:end_idx:stride]
        zw = z[start_idx:end_idx:stride]
        ww = w[start_idx:end_idx:stride]

        segs = make_segments(xw, yw, zw)
        lc.set_segments(segs)
        lc.set_array(ww[:-1])  # kleur per segment

        # optioneel: meeschuivende limieten
        if not fixed_limits:
            xmin2, xmax2 = np.min(xw), np.max(xw)
            ymin2, ymax2 = np.min(yw), np.max(yw)
            zmin2, zmax2 = np.min(zw), np.max(zw)
            pad = 0.1
            xr = xmax2 - xmin2 if xmax2 > xmin2 else 1.0
            yr = ymax2 - ymin2 if ymax2 > ymin2 else 1.0
            zr = zmax2 - zmin2 if zmax2 > zmin2 else 1.0
            ax.set_xlim(xmin2 - pad * xr, xmax2 + pad * xr)
            ax.set_ylim(ymin2 - pad * yr, ymax2 + pad * yr)
            ax.set_zlim(zmin2 - pad * zr, zmax2 + pad * zr)

        title.set_text(f"4D Rössler — t ≈ {t[start_idx]:.2f} … {t[end_idx-1]:.2f}  (kleur=w)")
        return (lc, title)

    anim = FuncAnimation(fig, update, frames=len(starts), interval=interval_ms, blit=False)

    # opslaan (optioneel)
    if save_path is not None:
        if save_path.lower().endswith(".mp4"):
            anim.save(save_path, writer="ffmpeg", fps=fps)
        elif save_path.lower().endswith(".gif"):
            anim.save(save_path, writer="pillow", fps=fps)
        else:
            raise ValueError("save_path moet eindigen op .mp4 of .gif")

    return anim


# -------------------------
# 4) Run demo
# -------------------------
if __name__ == "__main__":
    t, x, y, z, w = integrate_rossler4(
        t_span=(0.0, 400.0),
        n=120_000,
        s0=(1.0, 0.0, 0.0, 0.0),
        a=0.25, b=3.0, c=0.5, d=0.05
    )

    anim = animate_rossler4(
        t, x, y, z, w,
        skip_first_frac=0.2,
        window=6000,
        frame_step=80,
        interval_ms=30,
        stride=2,
        fixed_limits=True,
        save_path=None,   # bv "rossler4.mp4" (ffmpeg nodig) of "rossler4.gif"
        fps=30
    )

    plt.show()
